# Lab Orpheus ← Stage 1–2 manifest

Open-weight local (`orpheus-speech` + vLLM). **Không** API `orpheus-tts`.

**GPU:** A100 tốt; T4 (15GB) có thể chạy smoke `--limit 4` nhưng dễ chật.

**Lỗi cài thường gặp**
1. `libcudart.so.13` → vLLM/CUDA lệch → pin `vllm==0.7.3` + Restart
2. `numpy.dtype size changed` → NumPy ABI vỡ sau pip → Restart + cell cài (numpy==1.26.4 rồi vllm lại)

Repo: https://github.com/phamtu2x5/ielts-script2audio.git


In [1]:
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


In [2]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull origin {BRANCH}
%cd {WORKDIR}
assert Path("pyproject.toml").is_file(), "Not in repo root"
print("CWD:", os.getcwd())


Cloning into '/content/ielts-script2audio'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 208 (delta 95), reused 172 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (208/208), 141.86 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/ielts-script2audio
CWD: /content/ielts-script2audio


In [3]:
# === Install (Colab only; heavy packages stay off local laptop) ===
# Order matters. Common failures:
# 1) libcudart.so.13  → wrong vLLM CUDA wheel → pin vllm==0.7.3
# 2) numpy.dtype size changed → NumPy ABI break after pip pulls mixed wheels
#    → force one NumPy, then reinstall vllm/orpheus against it
#
# After ANY failed import: Runtime → Restart session → re-run THIS cell only first.

import sys
print("python", sys.version)

# Light project package first
!pip -q install -e ".[dev]" soundfile

# Remove paid API client if present
!pip -q uninstall -y orpheus-tts 2>/dev/null || true

# Stabilize NumPy ABI for scientific stack on Colab (1.26.x works with many older vLLM)
!pip -q install --force-reinstall "numpy==1.26.4"

# Orpheus local open-weight stack
!pip -q install "orpheus-speech"
!pip -q install --force-reinstall "vllm==0.7.3"

# If orpheus-speech pulled a different numpy, pin again
!pip -q install --force-reinstall "numpy==1.26.4"

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
    print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

import numpy as np
print("numpy", np.__version__)

try:
    import vllm
    from orpheus_tts import OrpheusModel
    print("vllm", getattr(vllm, "__version__", "?"))
    print("OrpheusModel import: OK")
except Exception as e:
    msg = str(e)
    hint = []
    if "libcudart.so.13" in msg:
        hint.append("CUDA wheel mismatch: keep vllm==0.7.3; Restart session; re-run this cell.")
    if "dtype size changed" in msg or "numpy" in msg.lower():
        hint.append(
            "NumPy ABI mismatch: Runtime→Restart session, then re-run this cell "
            "(forces numpy==1.26.4 then vllm==0.7.3)."
        )
    hint.append("Upstream: https://github.com/canopyai/Orpheus-TTS")
    hint.append(
        "Official Orpheus Colab (if still broken): "
        "https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N"
    )
    raise RuntimeError(
        "Orpheus/vLLM import failed.\n"
        + "\n".join(f"- {h}" for h in hint)
        + f"\nOriginal: {type(e).__name__}: {e}"
    ) from e


python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ielts-script2audio (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible

RuntimeError: Orpheus/vLLM import failed.
- Upstream: https://github.com/canopyai/Orpheus-TTS
- Official Orpheus Colab (if still broken): https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N
Original: ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

In [ ]:
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
Path("lab_audio/orpheus_part1").mkdir(parents=True, exist_ok=True)

!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

m = json.loads(Path("outputs/part1_manifest.json").read_text(encoding="utf-8"))
assert m["validation"]["valid"] is True, m["validation"].get("issues")
print("transcript_id", m["transcript_id"])
print("n_requests", len(m["requests"]))
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006", "e008", "e011"}:
        print(u["event_id"], "|", u["display_text"][:48], "||", u["spoken_text"][:64])


In [ ]:
import json
import os
from pathlib import Path

OUT_DIR = Path("lab_audio/orpheus_part1")
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = OUT_DIR / "lab_render_report.json"

# Smoke: --limit 4. Full dialogue: remove --limit 4
cmd = (
    "python scripts/lab_render_orpheus_from_manifest.py "
    "--manifest outputs/part1_manifest.json "
    "--voice-map configs/voice_maps/orpheus_en_part1.json "
    "--out-dir lab_audio/orpheus_part1 "
    "--temperature 0.6 "
    "--repetition-penalty 1.3 "
    "--limit 4"
)
print(cmd)
ret = os.system(cmd)
print("exit", ret)

if not REPORT.is_file():
    raise FileNotFoundError(
        f"{REPORT} missing — render failed (often CUDA/vLLM). "
        "Fix install cell, Restart session, re-run install→prepare→render."
    )

report = json.loads(REPORT.read_text(encoding="utf-8"))
print("ok", report["ok_count"], "fail", report["fail_count"], "model", report.get("model_name"))
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/orpheus_part1/lab_render_report.json")
assert REPORT_PATH.is_file(), "Run render cell first"
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent

print(report.get("transcript_id"), "n=", len(segments), report.get("backend"))
for i, seg in enumerate(segments, start=1):
    fname = seg.get("output_filename")
    wav = audio_dir / fname if fname else None
    print("=" * 72)
    print(
        f"[{i}/{len(segments)}] {seg.get('segment_id')} | "
        f"{seg.get('speaker_id')} | {seg.get('backend_voice_id')} | {seg.get('status')}"
    )
    print("DISPLAY:", seg.get("display_text", ""))
    print("SPOKEN :", seg.get("spoken_text", ""))
    if seg.get("error"):
        print("ERROR:", seg["error"])
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print("missing", fname)


In [ ]:
from pathlib import Path
import json

report_path = Path("lab_audio/orpheus_part1/lab_render_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))

reviews = {
    seg["segment_id"]: {"content_match": "", "notes": ""}
    for seg in (report.get("segments") or [])
    if seg.get("segment_id")
}
# After listen: reviews["seg_0001"]["content_match"] = "yes"  # yes|partial|no

filled = []
missing = []
for seg in report.get("segments") or []:
    sid = seg["segment_id"]
    human = reviews.get(sid) or {}
    match = (human.get("content_match") or "").strip().lower()
    if match not in {"yes", "partial", "no"}:
        missing.append(sid)
    filled.append(
        {
            "segment_id": sid,
            "speaker_id": seg.get("speaker_id"),
            "backend_voice_id": seg.get("backend_voice_id"),
            "display_text": seg.get("display_text"),
            "spoken_text": seg.get("spoken_text"),
            "output_filename": seg.get("output_filename"),
            "status": seg.get("status"),
            "content_match": match,
            "notes": human.get("notes", ""),
        }
    )

out = Path("lab_audio/orpheus_part1/segment_review_filled.json")
out.write_text(json.dumps(filled, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Wrote", out, "| unreviewed", missing)


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display

report = Path("lab_audio/orpheus_part1/lab_render_report.json")
assert report.is_file(), "Need successful render first"

!python scripts/lab_concat_segment_wavs.py \
  --report lab_audio/orpheus_part1/lab_render_report.json \
  --out lab_audio/orpheus_part1/part1_full.wav \
  --gap-mode adaptive \
  --same-speaker-gap-ms 220 \
  --turn-gap-ms 520

full = Path("lab_audio/orpheus_part1/part1_full.wav")
meta = json.loads(Path("lab_audio/orpheus_part1/part1_full.concat.json").read_text())
print(meta.get("duration_sec"), "sec", meta.get("num_segments_used"), "segs")
if full.is_file():
    display(Audio(filename=str(full)))


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json
from collections import defaultdict

!python scripts/lab_survey_orpheus_voices.py \
  --manifest outputs/part1_manifest.json \
  --inventory configs/voice_maps/orpheus_voice_inventory.json \
  --preset en_shortlist \
  --out-dir lab_audio/orpheus_voice_survey \
  --event-ids e004,e006,e008,e011 \
  --end-pad-ms 450

survey_report = Path("lab_audio/orpheus_voice_survey/voice_survey_report.json")
if not survey_report.is_file():
    raise FileNotFoundError("Survey failed — check Orpheus/vLLM install above")

report = json.loads(survey_report.read_text(encoding="utf-8"))
print("voices", report.get("voices"), "ok/fail", report.get("ok_count"), report.get("fail_count"))

by_event = defaultdict(list)
for r in report.get("renders") or []:
    by_event[r.get("event_id")].append(r)

for event_id, items in by_event.items():
    items = sorted(items, key=lambda x: x.get("voice_id") or "")
    sample = items[0] if items else {}
    print("=" * 72)
    print("LINE", event_id)
    print("DISPLAY:", sample.get("display_text"))
    print("SPOKEN :", sample.get("spoken_text"))
    for it in items:
        print("---", it.get("voice_id"), it.get("status"))
        if it.get("error"):
            print(it["error"])
            continue
        wav = Path("lab_audio/orpheus_voice_survey") / it["output_filename"]
        if wav.is_file():
            display(Audio(filename=str(wav)))


**Checklist:** import OK? 2 giọng tách? codes rõ? so Kokoro? không sửa script? Verdict lab only — `not_selected`.
